# Step 1 — Identify Training Vocabulary

**Goal:** For each dataset, extract all unique words and character bigrams (IV set) from transcriptions.  
This IV bigram set is the baseline — any word from a future corpus containing a bigram **not** in this set is considered OOV.

**Datasets:**
1. `Titung/cc100-nepali-tts`
2. `Titung/nepali-tts-tagged-combined`
3. `Titung/nepali-asr`

## 0. Install dependencies

In [ ]:
!pip install datasets -q

## 1. Imports

In [ ]:
from datasets import load_dataset, get_dataset_config_names
from collections import Counter
import json
import os

print("Imports OK")

## 2. Helper functions

In [ ]:
def extract_iv_sets(examples, text_field="text"):
    """
    Given an iterable of dataset examples, extract:
      - iv_words  : set of all unique whitespace-split tokens
      - iv_bigrams: set of all unique character bigrams across those tokens
    Returns (iv_words, iv_bigrams, word_freq, bigram_freq)
    """
    iv_words = set()
    iv_bigrams = set()
    word_freq = Counter()
    bigram_freq = Counter()

    for ex in examples:
        text = ex.get(text_field, "") or ""
        for word in text.split():
            word = word.strip()
            if not word:
                continue
            iv_words.add(word)
            word_freq[word] += 1
            for i in range(len(word) - 1):
                bg = word[i:i+2]
                iv_bigrams.add(bg)
                bigram_freq[bg] += 1

    return iv_words, iv_bigrams, word_freq, bigram_freq


def print_summary(name, iv_words, iv_bigrams, word_freq, bigram_freq):
    print(f"\n{'='*60}")
    print(f"Dataset : {name}")
    print(f"{'='*60}")
    print(f"  Unique words   : {len(iv_words):,}")
    print(f"  Unique bigrams : {len(iv_bigrams):,}")
    print(f"  Total word tokens (running count): {sum(word_freq.values()):,}")
    print(f"\n  Top 10 most frequent words:")
    for w, c in word_freq.most_common(10):
        print(f"    {w!r:25s}  {c:,}")
    print(f"\n  Top 10 most frequent bigrams:")
    for b, c in bigram_freq.most_common(10):
        print(f"    {b!r:10s}  {c:,}")


def save_result(name, iv_words, iv_bigrams, word_freq, bigram_freq, out_dir="step1_results"):
    os.makedirs(out_dir, exist_ok=True)
    slug = name.replace("/", "__")
    payload = {
        "dataset": name,
        "n_unique_words": len(iv_words),
        "n_unique_bigrams": len(iv_bigrams),
        "total_word_tokens": sum(word_freq.values()),
        "iv_words": sorted(iv_words),
        "iv_bigrams": sorted(iv_bigrams),
        "top100_words": word_freq.most_common(100),
        "top100_bigrams": bigram_freq.most_common(100),
    }
    path = os.path.join(out_dir, f"{slug}_iv.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"  → Saved to {path}")
    return path


print("Helpers defined.")

---
## 3. Dataset 1 — `Titung/cc100-nepali-tts`

In [ ]:
DATASET_1 = "Titung/cc100-nepali-tts"

# Inspect available configs / splits
try:
    configs = get_dataset_config_names(DATASET_1)
    print(f"Available configs: {configs}")
except Exception as e:
    print(f"No named configs (or error): {e}")
    configs = [None]

In [ ]:
# Load and inspect one example to find the text field name
ds1_peek = load_dataset(DATASET_1, split="train", trust_remote_code=True)
print("Column names:", ds1_peek.column_names)
print("First example:", ds1_peek[0])

In [ ]:
# ---- ADJUST this if the text field is named differently ----
TEXT_FIELD_1 = "text"   # e.g. "sentence", "transcript", "transcription"
# ------------------------------------------------------------

iv_words_1, iv_bigrams_1, word_freq_1, bigram_freq_1 = extract_iv_sets(
    ds1_peek, text_field=TEXT_FIELD_1
)

print_summary(DATASET_1, iv_words_1, iv_bigrams_1, word_freq_1, bigram_freq_1)
save_result(DATASET_1, iv_words_1, iv_bigrams_1, word_freq_1, bigram_freq_1)

---
## 4. Dataset 2 — `Titung/nepali-tts-tagged-combined`

In [ ]:
DATASET_2 = "Titung/nepali-tts-tagged-combined"

try:
    configs2 = get_dataset_config_names(DATASET_2)
    print(f"Available configs: {configs2}")
except Exception as e:
    print(f"No named configs (or error): {e}")
    configs2 = [None]

In [ ]:
ds2_peek = load_dataset(DATASET_2, split="train", trust_remote_code=True)
print("Column names:", ds2_peek.column_names)
print("First example:", ds2_peek[0])

In [ ]:
# ---- ADJUST if needed ----
TEXT_FIELD_2 = "text"
# --------------------------

iv_words_2, iv_bigrams_2, word_freq_2, bigram_freq_2 = extract_iv_sets(
    ds2_peek, text_field=TEXT_FIELD_2
)

print_summary(DATASET_2, iv_words_2, iv_bigrams_2, word_freq_2, bigram_freq_2)
save_result(DATASET_2, iv_words_2, iv_bigrams_2, word_freq_2, bigram_freq_2)

---
## 5. Dataset 3 — `Titung/nepali-asr`

In [ ]:
DATASET_3 = "Titung/nepali-asr"

try:
    configs3 = get_dataset_config_names(DATASET_3)
    print(f"Available configs: {configs3}")
except Exception as e:
    print(f"No named configs (or error): {e}")
    configs3 = [None]

In [ ]:
ds3_peek = load_dataset(DATASET_3, split="train", trust_remote_code=True)
print("Column names:", ds3_peek.column_names)
print("First example:", ds3_peek[0])

In [ ]:
# ---- ADJUST if needed ----
# ASR datasets often use 'sentence' or 'transcript' instead of 'text'
TEXT_FIELD_3 = "text"   # change to "sentence" / "transcript" if needed
# --------------------------

iv_words_3, iv_bigrams_3, word_freq_3, bigram_freq_3 = extract_iv_sets(
    ds3_peek, text_field=TEXT_FIELD_3
)

print_summary(DATASET_3, iv_words_3, iv_bigrams_3, word_freq_3, bigram_freq_3)
save_result(DATASET_3, iv_words_3, iv_bigrams_3, word_freq_3, bigram_freq_3)

---
## 6. Cross-dataset comparison

In [ ]:
all_results = [
    (DATASET_1, iv_words_1, iv_bigrams_1),
    (DATASET_2, iv_words_2, iv_bigrams_2),
    (DATASET_3, iv_words_3, iv_bigrams_3),
]

print("\n" + "="*60)
print("CROSS-DATASET COMPARISON")
print("="*60)
print(f"{'Dataset':<45}  {'Words':>8}  {'Bigrams':>8}")
print("-"*65)
for name, words, bigrams in all_results:
    print(f"{name:<45}  {len(words):>8,}  {len(bigrams):>8,}")

# Union of all bigrams (combined IV set)
union_bigrams = iv_bigrams_1 | iv_bigrams_2 | iv_bigrams_3
union_words   = iv_words_1   | iv_words_2   | iv_words_3
print("-"*65)
print(f"{'UNION (all three)':<45}  {len(union_words):>8,}  {len(union_bigrams):>8,}")

# Bigrams unique to each dataset (not in the others)
print("\nBigrams EXCLUSIVE to each dataset:")
for name, _, bigrams in all_results:
    others = set()
    for n2, _, b2 in all_results:
        if n2 != name:
            others |= b2
    exclusive = bigrams - others
    print(f"  {name}: {len(exclusive):,} exclusive bigrams  (sample: {sorted(exclusive)[:10]})")

---
## 7. Save the combined (union) IV set

This combined set will be the reference for Step 3 OOV detection.

In [ ]:
import os, json

os.makedirs("step1_results", exist_ok=True)

combined = {
    "description": "Union of IV words and bigrams across all three Titung datasets",
    "sources": [DATASET_1, DATASET_2, DATASET_3],
    "n_unique_words": len(union_words),
    "n_unique_bigrams": len(union_bigrams),
    "iv_words": sorted(union_words),
    "iv_bigrams": sorted(union_bigrams),
}

with open("step1_results/combined_iv.json", "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

print("Saved: step1_results/combined_iv.json")
print(f"  Combined IV words  : {len(union_words):,}")
print(f"  Combined IV bigrams: {len(union_bigrams):,}")

---
## Summary

Step 1 is complete. For each dataset you now have:

| Output file | Contents |
|---|---|
| `step1_results/Titung__cc100-nepali-tts_iv.json` | IV words + bigrams for dataset 1 |
| `step1_results/Titung__nepali-tts-tagged-combined_iv.json` | IV words + bigrams for dataset 2 |
| `step1_results/Titung__nepali-asr_iv.json` | IV words + bigrams for dataset 3 |
| `step1_results/combined_iv.json` | **Union** IV set — use this for Step 3 |

**Next:** Step 2 — Collate a large external corpus (e.g. CC-100 Nepali) to mine OOV word candidates.